In [30]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import rl4evrp as rl
from rl4evrp.environment import EVRPEnv


In [31]:
# Initialize framework with configs
framework = rl.RL4EVRP()

# Read configurations from YAML files
problem_config = framework.read_yaml('problem')
model_config = framework.read_yaml('model')
env_config = framework.read_yaml('env')

print("Problem Configuration:")
framework.config.print_config()

✓ Device: cpu
✓ Output directory: results_xai
Problem Configuration:

[ENV]
device: cuda
output_directory: results_xai
log_level: INFO
save_checkpoints: True
checkpoint_interval: 100
llm:
  enabled: False
  provider: groq
  model: llama3-8b-8192
  api_key: ${GROQ_API_KEY}
  timeout: 30
reward_tuning:
  charger_visit_penalty: 0.05
  early_return_penalty: 0.3
  service_bonus: 0.2
  completion_bonus: 2.0
  battery_violation_penalty: 1.0
diagnostics:
  track_masked_actions: True
  warn_on_violations: False
reproducibility:
  seed: 42
  deterministic: True

[MODEL]
encoder:
  type: gat
  embed_dim: 128
  n_heads: 8
  n_layers: 3
  ff_dim: 256
  dropout: 0.0
  layer_norm: True
decoder:
  type: cross_attention
  embed_dim: 128
  n_heads: 8
value_head:
  hidden_dim: 64
  activation: gelu
training:
  optimizer: adam
  lr: 0.0003
  epsilon: 1e-05
  weight_decay: 0.0
  scheduler: cosine_annealing
  T_max: 800
  eta_min_factor: 0.1
  grad_clip_norm: 1.0
  gamma: 0.99
  entropy_coefficient: 0.01
  

In [32]:
# ============================================================
# 1. Build framework and load model
# ============================================================

framework = rl.RL4EVRP()
model_builder = framework.build()

model = model_builder.complete_model()

✓ Device: cpu
✓ Output directory: results_xai
✓ Agent initialized with 488065 parameters


In [6]:
import numpy as np

# Generate training and evaluation instances
instance_cfg = problem_config['instance_generation']
n_train = instance_cfg.get('train_size', instance_cfg.get('n_train_instances', 50000))
n_eval = instance_cfg.get('eval_size', instance_cfg.get('n_eval_instances', 100))
seed_offset = instance_cfg.get('seed_offset', 0)

train_instances = [
    framework.generate_instance(seed=seed_offset + i)
    for i in range(n_train)
]

eval_instances = [
    framework.generate_instance(seed=1000 + seed_offset + i)
    for i in range(n_eval)
]

print(f"✓ Generated {len(train_instances)} training instances")
print(f"✓ Generated {len(eval_instances)} evaluation instances")

# Example instance
demo_inst = train_instances[0]
print(f"\nExample instance:")
print(f"  Nodes: {demo_inst['n_nodes']}")
print(f"  Chargers: {(demo_inst['node_types'] == 2).sum()}")
print(f"  Customers: {(demo_inst['node_types'] == 1).sum()}")

✓ Generated 250000 training instances
✓ Generated 50 evaluation instances

Example instance:
  Nodes: 26
  Chargers: 4
  Customers: 21


In [51]:
from rl4evrp.utils import train_agent

# Train for first seed
seeds = framework.get_seeds()
seed = seeds[0]

print(f"Training with seed: {seed}")

# Set seed
np.random.seed(seed)
import torch
torch.manual_seed(seed)

# Reinitialize model with seed
model = model_builder.complete_model()

# Train for N episodes (use small number for demo)
n_episodes = 1000 # Change to 800 for full training

training_results = train_agent(
    model,
    train_instances,
    n_episodes=n_episodes,
    device=str(framework.device),
    save_dir=framework.output_dir / 'ablation/checkpoints_50N_evrptw_1k',
    eval_instances=eval_instances,
    save_interval=10
)

print("✓ Training completed!")

# Save model
checkpoint_path = framework.output_dir / 'ablation/final_model_50N_evrptw_1k.pt'
torch.save(model.state_dict(), checkpoint_path)

print(f"✓ Model saved to {checkpoint_path}")

# Also save training config
import json
config_save_path = framework.output_dir / 'ablation/training_config_50N_evrptw_1k.json'
json.dump({
    'problem': problem_config,
    'model': model_config,
    'n_episodes': n_episodes,
    'seed': seed
}, open(config_save_path, 'w'), indent=2, default=str)

print(f"✓ Config saved to {config_save_path}")

Training with seed: 42
✓ Agent initialized with 488065 parameters
Episode 10: train_reward=-177.047, eval_reward=-97.644, loss=0.4245
Episode 20: train_reward=-182.505, eval_reward=-95.918, loss=0.4952
Episode 30: train_reward=-172.179, eval_reward=-154.664, loss=0.4874
Episode 40: train_reward=-178.766, eval_reward=-125.444, loss=0.4557
Episode 50: train_reward=-145.231, eval_reward=-125.444, loss=0.4416
Episode 60: train_reward=-161.160, eval_reward=-96.416, loss=0.4284
Episode 70: train_reward=-173.942, eval_reward=-96.416, loss=0.3415
Episode 80: train_reward=-177.127, eval_reward=-96.824, loss=0.4729
Episode 90: train_reward=-176.908, eval_reward=-125.858, loss=0.4741
Episode 100: train_reward=-174.985, eval_reward=-125.858, loss=0.4945
Episode 110: train_reward=-178.385, eval_reward=-148.624, loss=0.4871
Episode 120: train_reward=-177.665, eval_reward=-149.316, loss=0.4854
Episode 130: train_reward=-32.215, eval_reward=-122.322, loss=0.4935
Episode 140: train_reward=-182.473, eva

In [33]:
import importlib
import rl4evrp.environment as env_mod
importlib.reload(env_mod)

from rl4evrp.environment import EVRPEnv, build_node_features


test_inst = framework.generate_instance(seed=123)
test_inst["feature_mask"] = np.array([1,1,1,1,1,1,1,0,0,0], dtype=np.float32)

env = EVRPEnv(test_inst, reward_mode="distance")
obs = env.reset()

print(obs["node_features"][0])

tensor([0.6965, 0.2861, 0.0000, 0.0000, 1.0000, 0.6000, 0.5000, 0.0000, 0.0000,
        0.0000])


In [34]:
inst_full = framework.generate_instance(seed=123)
env_full = EVRPEnv(inst_full, reward_mode="distance")
obs_full = env_full.reset()

inst_masked = framework.generate_instance(seed=123)
inst_masked["feature_mask"] = np.array([1,1,1,1,1,1,1,0,0,0], dtype=np.float32)
env_masked = EVRPEnv(inst_masked, reward_mode="distance")
obs_masked = env_masked.reset()

print("Full first node:   ", obs_full["node_features"][0])
print("Masked first node: ", obs_masked["node_features"][0])

Full first node:    tensor([0.6965, 0.2861, 0.0000, 0.0000, 1.0000, 0.6000, 0.5000, 0.0000, 1.0000,
        0.0000])
Masked first node:  tensor([0.6965, 0.2861, 0.0000, 0.0000, 1.0000, 0.6000, 0.5000, 0.0000, 0.0000,
        0.0000])


In [22]:

model.load_state_dict(torch.load("agent_episode_100000.pt", map_location=framework.device))
model.eval()

RuntimeError: Error(s) in loading state_dict for A2CAgent:
	size mismatch for encoder.embed.weight: copying a param with shape torch.Size([128, 7]) from checkpoint, the shape in current model is torch.Size([128, 10]).
	size mismatch for decoder.proj_ctx.weight: copying a param with shape torch.Size([128, 130]) from checkpoint, the shape in current model is torch.Size([128, 131]).

In [72]:
# ============================================================
# 2. Helper: safely convert tensors
# ============================================================

def to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


# ============================================================
# 3. Helper: get action probabilities from model
# ============================================================

def get_action_probs(model, obs, valid_mask, device="cpu"):
    """
    Returns action probabilities for the current decision step.
    Assumes model._forward(obs) returns (scores, value, node_emb).
    """
    model.eval()

    with torch.no_grad():
        scores, value, node_emb = model._forward(obs)
        logits = scores.squeeze(0).clone()

        valid_mask_t = torch.tensor(valid_mask, dtype=torch.bool, device=logits.device)
        logits[~valid_mask_t] = -1e9

        probs = torch.softmax(logits, dim=-1)

    return np.array(probs.detach().cpu().tolist(), dtype=np.float32)


# ============================================================
# 4. Compute interpretable factors for each candidate
# ============================================================

def compute_candidate_factors(env, probs, valid_mask):
    """
    Build one row per candidate node for one decision step.

    Temporal urgency is only meaningful for customer nodes.
    For depot and charger nodes, time_slack is set to NaN.
    """
    inst = env.inst
    cur = env.cur
    node_types = np.array(inst["node_types"])
    demands = np.array(inst.get("demands", np.zeros(len(node_types))))

    current_time = float(getattr(env, "time", 0.0))
    current_battery = float(getattr(env, "battery", 0.0))

    ready_times = np.array(inst.get("ready_times", np.zeros(len(node_types))), dtype=np.float32)
    due_times = np.array(inst.get("due_times", np.zeros(len(node_types))), dtype=np.float32)
    service_times = np.array(inst.get("service_times", np.zeros(len(node_types))), dtype=np.float32)

    rows = []
    for j in range(len(node_types)):
        node_type = int(node_types[j])
        is_valid = bool(valid_mask[j])
        dist = float(env.D[cur, j])

        arrival_time = current_time + dist
        ready = float(ready_times[j]) if j < len(ready_times) else 0.0
        due = float(due_times[j]) if j < len(due_times) else 0.0
        service_time = float(service_times[j]) if j < len(service_times) else 0.0

        waiting_time = max(0.0, ready - arrival_time)
        start_service = max(arrival_time, ready)

        # Temporal urgency is meaningful only for customer nodes
        if node_type == 1:  # customer
            slack = due - start_service
            temporal_applicable = 1
        else:
            slack = np.nan
            temporal_applicable = 0

        battery_after_travel = current_battery - dist

        rows.append({
            "candidate": j,
            "valid": int(is_valid),
            "selected_prob": float(probs[j]),
            "distance": dist,
            "arrival_time": float(arrival_time),
            "ready_time": ready,
            "due_time": due,
            "waiting_time": float(waiting_time),
            "time_slack": float(slack) if not np.isnan(slack) else np.nan,
            "battery_after_travel": float(battery_after_travel),
            "node_type": node_type,
            "is_customer": int(node_type == 1),
            "is_charger": int(node_type == 2),
            "is_depot": int(node_type == 0),
            "temporal_applicable": temporal_applicable,
            "demand": float(demands[j]) if j < len(demands) else 0.0,
            "service_time": service_time,
        })

    df = pd.DataFrame(rows)
    return df.sort_values(["valid", "selected_prob"], ascending=[False, False]).reset_index(drop=True)


# ============================================================
# 5. Run one episode and capture one decision step
# ============================================================

def decision_attribution_case_study(model, instance, feature_names, step_of_interest=3, device="cpu"):
    """
    Run greedily until step_of_interest, then analyze that decision.
    """
    inst = dict(instance, reward_mode="distance")
    inst["state_feature_names"] = list(feature_names)

    env = EVRPEnv(inst, reward_mode="distance")

    obs = env.reset()
    done = False
    step = 0

    while not done:
        valid_mask = env._valid_mask()
        probs = get_action_probs(model, obs, valid_mask, device=device)

        if step == step_of_interest:
            df = compute_candidate_factors(env, probs, valid_mask)
            selected_action = int(np.argmax(np.where(valid_mask, probs, -1.0)))
            df["selected"] = (df["candidate"] == selected_action).astype(int)

            context = {
                "step": step,
                "current_node": int(env.cur),
                "route_so_far": list(env.route) if hasattr(env, "route") else [],
                "current_time": float(getattr(env, "time", getattr(env, "t", 0.0))),
                "current_battery": float(getattr(env, "battery", getattr(env, "battery_level", 0.0))),
                "selected_action": selected_action,
                "feature_names": list(feature_names),
            }
            return context, df

        action = int(np.argmax(np.where(valid_mask, probs, -1.0)))
        obs, reward, done, info = env.step(action)
        step += 1

    raise ValueError(f"Episode ended before reaching step {step_of_interest}.")


# ============================================================
# 6. Plot attribution figure
# ============================================================

def plot_decision_attribution(context, df, save_path="decision_attribution_case.pdf", top_k=8):
    """
    Plot a compact figure for the top valid candidates.
    Temporal urgency is shown only for customer nodes.
    Depot/charger bars are drawn at zero in gray in panel (c).
    """
    df_valid = df[df["valid"] == 1].copy()
    df_valid = df_valid.sort_values("selected_prob", ascending=False).head(top_k).copy()

    def label_row(row):
        nid = int(row["candidate"])
        if row["is_depot"]:
            t = "D"
        elif row["is_charger"]:
            t = "CS"
        else:
            t = "Cust"
        return f"{nid} ({t})"

    df_valid["label"] = df_valid.apply(label_row, axis=1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))

    # ---------- Panel 1: policy probability ----------
    bars = axes[0].bar(df_valid["label"], df_valid["selected_prob"])
    for i, (_, row) in enumerate(df_valid.iterrows()):
        if row["selected"] == 1:
            bars[i].set_linewidth(2.0)
            bars[i].set_edgecolor("black")
    axes[0].set_title("(a) Policy preference", loc="left", fontweight="bold")
    axes[0].set_ylabel("Action probability")
    axes[0].tick_params(axis="x", rotation=35)

    # ---------- Panel 2: distance ----------
    bars = axes[1].bar(df_valid["label"], df_valid["distance"])
    for i, (_, row) in enumerate(df_valid.iterrows()):
        if row["selected"] == 1:
            bars[i].set_linewidth(2.0)
            bars[i].set_edgecolor("black")
    axes[1].set_title("(b) Spatial cost", loc="left", fontweight="bold")
    axes[1].set_ylabel("Distance from current node")
    axes[1].tick_params(axis="x", rotation=35)

    # ---------- Panel 3: temporal urgency ----------
    # Customer nodes use real slack; non-customer nodes shown as 0 in gray
    slack_plot_vals = df_valid["time_slack"].fillna(0.0).values
    temporal_colors = [
        "tab:blue" if row["temporal_applicable"] == 1 else "lightgray"
        for _, row in df_valid.iterrows()
    ]

    bars = axes[2].bar(df_valid["label"], slack_plot_vals, color=temporal_colors)
    for i, (_, row) in enumerate(df_valid.iterrows()):
        if row["selected"] == 1:
            bars[i].set_linewidth(2.0)
            bars[i].set_edgecolor("black")

    axes[2].axhline(0, linestyle="--", linewidth=1)
    axes[2].set_title("(c) Temporal urgency", loc="left", fontweight="bold")
    axes[2].set_ylabel("Time slack (due - service start)")
    axes[2].tick_params(axis="x", rotation=35)

    title = (
        f"Decision attribution at step {context['step']} | "
        f"current node={context['current_node']} | "
        f"selected={context['selected_action']} | "
        f"time={context['current_time']:.2f} | "
        f"battery={context['current_battery']:.2f}"
    )
    fig.suptitle(title, fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure to {save_path}")


# ============================================================
# 7. Save table for the paper / appendix
# ============================================================

def save_attribution_table(df, save_csv="decision_attribution_case.csv"):
    cols = [
        "candidate", "valid", "selected", "selected_prob",
        "distance", "time_slack", "battery_after_travel",
        "is_customer", "is_charger", "is_depot", "temporal_applicable",
        "demand", "arrival_time", "ready_time", "due_time",
        "waiting_time", "service_time"
    ]
    out = df[cols].sort_values(["valid", "selected_prob"], ascending=[False, False])
    out.to_csv(save_csv, index=False)
    print(f"Saved table to {save_csv}")

In [44]:
def load_model(model_path):
    framework = rl.RL4EVRP()
    framework.read_yaml("problem")
    framework.read_yaml("model")
    framework.read_yaml("env")

    model_builder = framework.build()
    model = model_builder.complete_model()

    state_dict = torch.load(model_path, map_location=framework.device)

    print("Checkpoint encoder shape:", state_dict["encoder.embed.weight"].shape)
    print("Current model encoder shape:", model.encoder.embed.weight.shape)

    print("Checkpoint decoder ctx shape:", state_dict["decoder.proj_ctx.weight"].shape)
    print("Current model decoder ctx shape:", model.decoder.proj_ctx.weight.shape)

    model.load_state_dict(state_dict)
    model.to(framework.device)
    model.eval()

    return framework, model

In [68]:
from rl4evrp.agents import A2CAgent
import rl4evrp.agents as agent_mod
import rl4evrp.models as models_mod

importlib.reload(models_mod)
importlib.reload(agent_mod)

from rl4evrp.agents import A2CAgent
from rl4evrp.models import EVRPEncoder, EVRPDecoder

def load_model_adaptive(model_path):
    framework = rl.RL4EVRP()

    state_dict = torch.load(model_path, map_location="cpu")
    in_dim = state_dict["encoder.embed.weight"].shape[1]
    ctx_dim = state_dict["decoder.proj_ctx.weight"].shape[1]

    print("Checkpoint signature:")
    print(f"  encoder input dim = {in_dim}")
    print(f"  decoder ctx dim   = {ctx_dim}")

    framework.read_yaml("problem")
    framework.read_yaml("model")
    framework.read_yaml("env")

    if in_dim == 7 and ctx_dim == 130:
        feature_names = [
            "x",
            "y",
            "demand_normalized",
            "is_charger",
            "is_depot",
            "cargo_capacity_norm",
            "battery_capacity_norm",
        ]
        use_time_context = False

    elif in_dim == 10 and ctx_dim == 131:
        feature_names = [
            "x",
            "y",
            "demand_normalized",
            "is_charger",
            "is_depot",
            "cargo_capacity_norm",
            "battery_capacity_norm",
            "tw_ready_norm",
            "tw_due_norm",
            "service_time_norm",
        ]
        use_time_context = True

    else:
        raise ValueError(
            f"Unsupported checkpoint signature: input_dim={in_dim}, ctx_dim={ctx_dim}"
        )

    # Build model directly using the agent class
    model = A2CAgent(
        input_dim=in_dim,
        embed_dim=128,
        n_heads=8,
        n_layers=3,
        enc_type="gat",
        device=str(framework.device),
        use_time_context=use_time_context,
    )

    print("Current model signature:")
    print(f"  encoder.embed.weight = {tuple(model.encoder.embed.weight.shape)}")
    print(f"  decoder.proj_ctx.weight = {tuple(model.decoder.proj_ctx.weight.shape)}")

    model.load_state_dict(state_dict)
    model.to(framework.device)
    model.eval()

    return framework, model, feature_names

In [78]:

# ============================================================
# 8. Example run
# ============================================================

framework, model, feature_names = load_model_adaptive("agent_episode_26800.pt")

#model = load_model("agent_evrptw_60k.pt")
#print(f"✓ Model loaded on device: {framework.device}")

seed = 42
instance = framework.generate_instance(seed=seed)

# For old EVRP checkpoint: no time windows
instance["tw_enabled"] = False
instance["ready_times"] = np.zeros(instance["n_nodes"], dtype=np.float32)
instance["due_times"] = np.full(instance["n_nodes"], instance["time_horizon"], dtype=np.float32)
instance["service_times"] = np.zeros(instance["n_nodes"], dtype=np.float32)

context, df = decision_attribution_case_study(
    model=model,
    instance=instance,
    feature_names=feature_names,
    step_of_interest=3,
    device=str(framework.device),
)

print("Context:")
for k, v in context.items():
    print(f"  {k}: {v}")

print("\nTop candidates:")
print(df.head(10))

plot_decision_attribution(
    context,
    df,
    save_path="decision_attribution_case.pdf",
    top_k=8,
)

save_attribution_table(df, save_csv="decision_attribution_case.csv")

✓ Device: cpu
✓ Output directory: results_xai
Checkpoint signature:
  encoder input dim = 7
  decoder ctx dim   = 130
Current model signature:
  encoder.embed.weight = (128, 7)
  decoder.proj_ctx.weight = (128, 130)
Context:
  step: 3
  current_node: 14
  route_so_far: [0, 25, 1, 14]
  current_time: 1.4859443306922913
  current_battery: 98.5140609741211
  selected_action: 10
  feature_names: ['x', 'y', 'demand_normalized', 'is_charger', 'is_depot', 'cargo_capacity_norm', 'battery_capacity_norm']

Top candidates:
   candidate  valid  selected_prob  distance  arrival_time  ready_time  \
0         10      1       0.385506  0.095052      1.580997         0.0   
1         19      1       0.360043  0.404267      1.890212         0.0   
2         15      1       0.192639  0.124993      1.610937         0.0   
3          4      1       0.044353  0.661679      2.147624         0.0   
4          9      1       0.016379  0.292689      1.778634         0.0   
5         12      1       0.000669  0.

C:\Users\dnouicer\AppData\Local\Temp\ipykernel_30568\1314499467.py:26: DeprecationWarning: In future, it will be an error for 'np.bool' scalars to be interpreted as an index
  valid_mask_t = torch.tensor(valid_mask, dtype=torch.bool, device=logits.device)


Saved figure to decision_attribution_case.pdf
Saved table to decision_attribution_case.csv
